# Setup
In colab:

Go to "Runtime" -> "Change runtime type" -> Select "T4 GPU"
Install TerraTorch

In [1]:
%%time
# ── 1. Python deps ────────────────────────────────────────────────────────────
# terratorch 1.0.1 is what the notebooks were written against
# git-lfs is needed because the GeoTIFFs in examples/ are stored with LFS
!pip install --upgrade pip
!pip install terratorch==1.0.1 gdown git+https://github.com/huggingface/huggingface_hub
!pip install diffusers==0.30.0
!pip install torch

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/huggingface/huggingface_hub to /tmp/pip-req-build-ce83qgbm
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/huggingface_hub /tmp/pip-req-build-ce83qgbm
  Resolved https://github.com/huggingface/huggingface_hub to commit 8318fe4e8791621ddb33bb3bd7c0d99d3eda53e4
  Installing build dependencies ... one
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
CPU times: user 117 ms, sys: 11.6 ms, total: 129 ms
Wall time: 11.3 s


In [2]:
%%time 
# ── 4. Import Libraries relevant to this notebook ─────────────────────────────
import torch
import numpy as np
import rioxarray as rxr
import matplotlib.pyplot as plt
from terratorch import FULL_MODEL_REGISTRY
from terratorch.models.backbones.terramind.model.terramind_register import v1_pretraining_mean, v1_pretraining_std
import diffusers

CPU times: user 5.11 s, sys: 1.21 s, total: 6.33 s
Wall time: 20.1 s


In [3]:
%%time
# ── 5. Select device ──────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
device

CPU times: user 71 μs, sys: 17 μs, total: 88 μs
Wall time: 92.5 μs


'cpu'

In [4]:
import sys
print(sys.executable)


/app/terramind/venv/bin/python


# Tokenizer Reconstruction
TerraMind includes internal tokenizers that convert raw geospatial data into latent tokens for learning and generation. In some use cases—especially for debugging, interpretation, or downstream model input—it is valuable to reconstruct and visualize the decoded data from these tokens. The Tokenizer Reconstructor enables this by reversing the tokenization process, mapping learned latent tokens back to spatially aligned, human-interpretable modalities such as satellite images or land cover maps.

This example demonstrates how to use the reconstruct_from_tokens function in TerraTorch to decode intermediate representations into full-resolution outputs.

In [5]:
import sys
!{sys.executable} -m pip install diffusers==0.30.0


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /app/terramind/venv/bin/python -m pip install --upgrade pip


In [ ]:
import sys
print(sys.executable)

In [ ]:
%%time
model = FULL_MODEL_REGISTRY.build('terramind_v1_tokenizer_s2l2a', pretrained=True)
model.to(device)

TerraMind_Tokenizer_S2L2A.pt:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

In [ ]:
%%time
# Build model
model = FULL_MODEL_REGISTRY.build('terramind_v1_tokenizer_s2l2a', pretrained=True)

# For other modalities:
# model = FULL_MODEL_REGISTRY.build('terramind_v1_tokenizer_s1rtc', pretrained=True)
# model = FULL_MODEL_REGISTRY.build('terramind_v1_tokenizer_dem', pretrained=True)
# model = FULL_MODEL_REGISTRY.build('terramind_v1_tokenizer_lulc', pretrained=True)
# model = FULL_MODEL_REGISTRY.build('terramind_v1_tokenizer_ndvi', pretrained=True)

_ = model.to(device)

In [ ]:
%%time
# Load an example (Replace S2L2A in the file paths for other modalities) 
examples = [
    '../examples/S2L2A/38D_378R_2_3.tif',
    '../examples/S2L2A/282D_485L_3_3.tif',
    '../examples/S2L2A/433D_629L_3_1.tif',
    '../examples/S2L2A/637U_59R_1_3.tif',
    '../examples/S2L2A/609U_541L_3_0.tif',
]

# Select example between 0 and 4
data = rxr.open_rasterio(examples[1])
# Conver to shape [B, C, 224, 224]
data = torch.Tensor(data.values, device='cpu').unsqueeze(0)

In [ ]:
%%time
# Visualize S-2 L2A input as RGB
rgb = data[0, [3,2,1]].clone().permute(1,2,0)
rgb = (rgb / 2000).clip(0, 1) * 255
rgb = rgb.cpu().numpy().round().astype(np.uint8)
plt.imshow(rgb)
plt.axis('off')
plt.show()

In [ ]:
%%time
# Normalize input
mean = torch.Tensor(v1_pretraining_mean['untok_sen2l2a@224'])
std = torch.Tensor(v1_pretraining_std['untok_sen2l2a@224'])
input = (data - mean[None, :, None, None]) / std[None, :, None, None]

# See keys for other modalities:
# v1_pretraining_mean.keys()

In [ ]:
%%time
# Run model with diffusion steps
input = input.to(device)
with torch.no_grad():
    # Encode & decode image
    reconstruction = model(input, timesteps=10)

    # Alternatively split the encoding and decoding process to analyze tokens 
    # Encode image
    # _, _, tokens = model.encode(input)
    # Decode tokens
    # reconstruction = model.decode_tokens(tokens, verbose=True, timesteps=10)

# Denormalize
reconstruction = reconstruction.cpu()
reconstruction = (reconstruction * std[None, :, None, None]) + mean[None, :, None, None]

In [ ]:
%%time
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

# Visualize S-2 L2A input as RGB
rgb = data[0, [3,2,1]].clone().permute(1,2,0)
rgb = (rgb / 2000).clip(0, 1) * 255
rgb = rgb.cpu().numpy().round().astype(np.uint8)
ax[0].imshow(rgb)
ax[0].axis('off')
ax[0].set_title('Input')

# Visualize S-2 L2A reconstruction as RGB
rgb = reconstruction[0, [3,2,1]].clone().permute(1,2,0)
rgb = (rgb / 2000).clip(0, 1) * 255
rgb = rgb.cpu().numpy().round().astype(np.uint8)
ax[1].imshow(rgb)
ax[1].axis('off')
ax[1].set_title('Reconstruction')

plt.show()